In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Input, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Load and visualize
data = pd.read_csv("daily-min-temperatures.csv")   # <-- put the CSV in the same folder / upload to Colab
temps = data["Temp"].values
plt.figure(figsize=(12,4))
plt.plot(temps)
plt.title("Daily Minimum Temperatures in Melbourne")
plt.xlabel("Days")
plt.ylabel("Temperature")
plt.show()

# 2. Normalize
temps = (temps - temps.min()) / (temps.max() - temps.min())

# 3. Create sequences
timesteps = 10
X, y = [], []
for i in range(len(temps) - timesteps):
    X.append(temps[i:i+timesteps])
    y.append(temps[i+timesteps])
X = np.array(X).reshape((len(X), timesteps, 1))
y = np.array(y)

# 4. Train-test split
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 5. RNN
model_rnn = Sequential([SimpleRNN(32, input_shape=(timesteps, 1)), Dense(1)])
model_rnn.compile(optimizer='adam', loss='mse')
history_rnn = model_rnn.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32, verbose=1)

# 6. LSTM
model_lstm = Sequential([LSTM(50, input_shape=(timesteps, 1)), Dense(1)])
model_lstm.compile(optimizer='adam', loss='mse')
history_lstm = model_lstm.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32, verbose=1)

# 7. Simple Transformer
def transformer_block(inputs):
    attention = MultiHeadAttention(num_heads=2, key_dim=8)(inputs, inputs)
    x = LayerNormalization()(inputs + attention)
    ffn = Dense(32, activation='relu')(x)
    ffn = Dense(1)(ffn)                    # note: the lab PDF has this
    return LayerNormalization()(x + ffn)

inputs = Input(shape=(timesteps, 1))
x = transformer_block(inputs)
x = GlobalAveragePooling1D()(x)
outputs = Dense(1)(x)
model_tr = Model(inputs, outputs)
model_tr.compile(optimizer='adam', loss='mse')
history_tr = model_tr.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32, verbose=1)

# 8. Evaluation function
def evaluate(model, name):
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mse)
    print(f"{name} -> MSE: {mse:.5f}, MAE: {mae:.5f}, RMSE: {rmse:.5f}")
    return mse, mae, rmse

print("=== Baseline Results ===")
evaluate(model_rnn, "RNN")
evaluate(model_lstm, "LSTM")
evaluate(model_tr, "Transformer")

# 9. Visualization (actual vs prediction - LSTM & Transformer)
pred_lstm = model_lstm.predict(X_test)
pred_tr = model_tr.predict(X_test)
plt.figure(figsize=(12,5))
plt.plot(y_test[:200], label="Actual", color="black")
plt.plot(pred_lstm[:200], label="LSTM")
plt.plot(pred_tr[:200], label="Transformer")
plt.legend()
plt.title("Model Comparison (first 200 test points)")
plt.show()